In [ ]:
import jax
print(jax.devices())

[TpuDevice(id=0, process_index=0, coords=(0,0,0), core_on_chip=0)]


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [1]:
!nvidia-smi

Fri Jun 19 20:50:48 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   33C    P0             52W /  400W |       0MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

## Task 1: Understanding MaxText Dataset Formats

MaxText supports multiple dataset formats for training large language models.
The main formats are:

### 1. Synthetic Dataset (`dataset_type=synthetic`)

* Generates random tokens on the fly.
* No external dataset is required.
* Primarily used for benchmarking, debugging, and performance testing.
* No preprocessing required.
* Ideal for testing CPU, GPU, and TPU performance.
* Reproducible and lightweight.

Cons:
* Not meaningful training data.
* Loss values do not reflect real-world language learning.
* Cannot be used to evaluate model quality.

### 2. TFDS Dataset (`dataset_type=tfds`)

Description

* Uses datasets from TensorFlow Datasets (TFDS).
* Easy access to standardized datasets.
* Well-integrated with TensorFlow and JAX pipelines.
* Suitable for real model training.

Cons:
* Requires dataset download and preprocessing.
* Higher storage requirements.
* Slower setup compared to synthetic data.


### 3. Grain Dataset (`dataset_type=grain`)

Description

* Uses Google's Grain data loading framework.
* Designed for large-scale distributed training.
* High throughput.
* Efficient shuffling and preprocessing.
* Scales well across multiple accelerators.
* More complex setup.


In [3]:
!pip install uv
!uv pip install maxtext[tpu]==0.2.1 --system --resolution=lowest
!install_tpu_pre_train_extra_deps

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.4/25.4 MB 104.3 MB/s eta 0:00:00
Using Python 3.12.13 environment at: /usr
Resolved 258 packages in 1.54s
Prepared 95 packages in 21.58s
Uninstalled 27 packages in 288ms
Installed 95 packages in 128ms
 - absl-py==1.4.0
 + absl-py==2.3.1
 - aiofiles==24.1.0
 + aiofiles==25.1.0
 + aqtp==0.9.0
 + astroid==4.0.2
 + auditwheel==6.5.0
 + black==24.10.0
 + build==1.3.0
 + cfgv==3.5.0
 + cheroot==11.1.2
 + chex==0.1.91
 + cloud-accelerator-diagnostics==0.1.1
 + cloud-tpu-diagnostics==0.1.5
 + clu==0.0.12
 + colorama==0.4.6
 + coverage==7.12.0
 + dacite==1.9.2
 + dataclasses==0.5
 + dataclasses-json==0.0.1
 - datasets==4.0.0
 + datasets==4.4.1
 - decorator==4.4.2
 + decorator==5.2.1
 - dill==0.3.8
 + dill==0.4.0
 + distlib==0.4.0
 + drjax==0.1.4
 + einshape==1.0
 + evaluate==0.4.6
 + execnet==2.1.2
 - flax==0.11.2
 + flax==0.12.6
 - fsspec==2025.3.0
 + fsspec==2025.10.0
 - gcsfs==2025.3.0
 + gcsfs==2025.10.0
 + google-cloud-mldiagnostics==0.5.10
 +

In [4]:
!uv pip install maxtext[cuda12]==0.2.1 --system --resolution=lowest

Using Python 3.12.13 environment at: /usr
Resolved 263 packages in 281ms
Prepared 14 packages in 3m 19s
Uninstalled 11 packages in 25ms
Installed 14 packages in 15ms
 - jax-cuda12-pjrt==0.7.2
 + jax-cuda12-pjrt==0.8.1
 - jax-cuda12-plugin==0.7.2
 + jax-cuda12-plugin==0.8.1
 - nvidia-cublas-cu12==12.8.4.1
 + nvidia-cublas-cu12==12.9.1.4
 - nvidia-cuda-cupti-cu12==12.8.90
 + nvidia-cuda-cupti-cu12==12.9.79
 - nvidia-cuda-nvcc-cu12==12.8.93
 + nvidia-cuda-nvcc-cu12==12.9.86
 - nvidia-cuda-nvrtc-cu12==12.8.93
 + nvidia-cuda-nvrtc-cu12==12.9.86
 - nvidia-cuda-runtime-cu12==12.8.90
 + nvidia-cuda-runtime-cu12==12.9.79
 - nvidia-cufft-cu12==11.3.3.83
 + nvidia-cufft-cu12==11.4.1.4
 - nvidia-cusolver-cu12==11.7.3.90
 + nvidia-cusolver-cu12==11.7.5.82
 - nvidia-cusparse-cu12==12.5.8.93
 + nvidia-cusparse-cu12==12.5.10.65
 - nvidia-nvjitlink-cu12==12.8.93
 + nvidia-nvjitlink-cu12==12.9.86
 + transformer-engine==2.9.0
 + transformer-engine-cu12==2.9.0
 + transformer-engine-jax==2.9.0


In [5]:
!install_cuda12_pre_train_extra_deps

/bin/bash: line 1: install_cuda12_pre_train_extra_deps: command not found


In [7]:
!python -c "import maxtext, pkgutil; print(list(maxtext.__path__))"

['/usr/local/lib/python3.12/dist-packages/maxtext']


In [8]:
!find /usr/local/lib/python3.12/dist-packages/maxtext -name "base.yml"

/usr/local/lib/python3.12/dist-packages/maxtext/configs/base.yml


In [9]:
!TF_GPU_ALLOCATOR=cuda_malloc_async python -m maxtext.trainers.pre_train.train \
  /usr/local/lib/python3.12/dist-packages/maxtext/configs/base.yml \
  model_name=qwen3-1.7b \
  dataset_type=synthetic \
  steps=5 \
  run_name=test_a100 \
  base_output_directory=/content/maxtext_outputs \
  skip_jax_distributed_system=True

W0619 21:00:42.717390 134415228277376 pyconfig.py:211] tokenizer_path not found in HF_IDS in maxtext/src/maxtext/utils/globals.py.           Using the default src/maxtext/assets/tokenizers/tokenizer.llama2 instead.           Please pass tokenizer_path in your command if this is not intended.
I0619 21:00:42.717758 134415228277376 max_utils.py:197] Skipping jax distributed system due to skip_jax_distributed_system=True flag.
Failed to find host bounds for accelerator type: WARNING: could not determine TPU accelerator type, please set env var `TPU_ACCELERATOR_TYPE` manually, otherwise libtpu.so may not properly initialize.
E0000 00:00:1781902842.850389    6148 common_lib.cc:530] INVALID_ARGUMENT: Error: unexpected worker hostname 'WARNING: could not determine TPU worker hostnames or IP addresses' from env var TPU_WORKER_HOSTNAMES. Expecting a valid hostname or IP address without port number. (Full TPU workers' addr string: WARNING: could not determine TPU worker hostnames or IP addresses,

In [10]:
!TF_GPU_ALLOCATOR=cuda_malloc_async python -m maxtext.trainers.pre_train.train \
  /usr/local/lib/python3.12/dist-packages/maxtext/configs/base.yml \
  model_name=qwen3-1.7b \
  dataset_type=synthetic \
  steps=50 \
  run_name=qwen_1p7b_gpu_a100_50steps \
  base_output_directory=/content/maxtext_outputs \
  skip_jax_distributed_system=True \
  checkpoint_period=100000 \
  log_period=1

W0619 21:06:15.511100 132772539884160 pyconfig.py:211] tokenizer_path not found in HF_IDS in maxtext/src/maxtext/utils/globals.py.           Using the default src/maxtext/assets/tokenizers/tokenizer.llama2 instead.           Please pass tokenizer_path in your command if this is not intended.
I0619 21:06:15.511464 132772539884160 max_utils.py:197] Skipping jax distributed system due to skip_jax_distributed_system=True flag.
Failed to find host bounds for accelerator type: WARNING: could not determine TPU accelerator type, please set env var `TPU_ACCELERATOR_TYPE` manually, otherwise libtpu.so may not properly initialize.
E0000 00:00:1781903175.623906    8327 common_lib.cc:530] INVALID_ARGUMENT: Error: unexpected worker hostname 'WARNING: could not determine TPU worker hostnames or IP addresses' from env var TPU_WORKER_HOSTNAMES. Expecting a valid hostname or IP address without port number. (Full TPU workers' addr string: WARNING: could not determine TPU worker hostnames or IP addresses,

In [11]:
!mkdir -p /content/assignment_logs
!cp -r /content/maxtext_outputs/qwen_1p7b_gpu_a100_50steps /content/assignment_logs/

In [13]:
!find /usr/local/lib/python3.12/dist-packages/maxtext/configs -name "*qwen*"

/usr/local/lib/python3.12/dist-packages/maxtext/configs/models/qwen3-0.6b.yml
/usr/local/lib/python3.12/dist-packages/maxtext/configs/models/qwen3-1.7b.yml
/usr/local/lib/python3.12/dist-packages/maxtext/configs/models/qwen2.5-14b.yml
/usr/local/lib/python3.12/dist-packages/maxtext/configs/models/qwen3-1.7b-base.yml
/usr/local/lib/python3.12/dist-packages/maxtext/configs/models/qwen3-32b.yml
/usr/local/lib/python3.12/dist-packages/maxtext/configs/models/qwen3-4b.yml
/usr/local/lib/python3.12/dist-packages/maxtext/configs/models/qwen2.5-7b.yml
/usr/local/lib/python3.12/dist-packages/maxtext/configs/models/qwen3-4b-thinking-2507.yml
/usr/local/lib/python3.12/dist-packages/maxtext/configs/models/qwen3-4b-base.yml
/usr/local/lib/python3.12/dist-packages/maxtext/configs/models/qwen3-8b.yml
/usr/local/lib/python3.12/dist-packages/maxtext/configs/models/qwen3-30b-a3b.yml
/usr/local/lib/python3.12/dist-packages/maxtext/configs/models/qwen3-30b-a3b-base.yml
/usr/local/lib/python3.12/dist-packag

In [14]:
!TF_GPU_ALLOCATOR=cuda_malloc_async python -m maxtext.trainers.pre_train.train \
  /usr/local/lib/python3.12/dist-packages/maxtext/configs/base.yml \
  model_name=qwen3-0.6b \
  dataset_type=synthetic \
  steps=50 \
  run_name=qwen_0p6b_gpu_a100_50steps \
  base_output_directory=/content/maxtext_outputs \
  skip_jax_distributed_system=True \
  checkpoint_period=100000 \
  log_period=1

I0619 21:24:45.673931 135360735466112 max_utils.py:197] Skipping jax distributed system due to skip_jax_distributed_system=True flag.
Failed to find host bounds for accelerator type: WARNING: could not determine TPU accelerator type, please set env var `TPU_ACCELERATOR_TYPE` manually, otherwise libtpu.so may not properly initialize.
E0000 00:00:1781904285.828075   13410 common_lib.cc:530] INVALID_ARGUMENT: Error: unexpected worker hostname 'WARNING: could not determine TPU worker hostnames or IP addresses' from env var TPU_WORKER_HOSTNAMES. Expecting a valid hostname or IP address without port number. (Full TPU workers' addr string: WARNING: could not determine TPU worker hostnames or IP addresses, please set env var `TPU_WORKER_HOSTNAMES` manually, otherwise libtpu.so may not properly initialize.)
=== Source Location Trace: === 
learning/45eac/tfrc/runtime/libtpu_init_utils.cc:287
INFO:2026-06-19 21:24:45,858:jax._src.xla_bridge:812: Unable to initialize backend 'tpu': UNKNOWN: TPU in

In [15]:
!ls /content/maxtext_outputs

qwen_0p6b_gpu_a100_50steps  qwen_1p7b_gpu_a100_50steps	test_a100


In [26]:
import os, shutil

os.makedirs("/content/submission", exist_ok=True)

for run in [
    "qwen_0p6b_gpu_a100_50steps",
    "qwen_1p7b_gpu_a100_50steps"
]:
    src = f"/content/maxtext_outputs/{run}"
    dst = f"/content/submission/{run}"

    if os.path.exists(src):
        os.makedirs(dst, exist_ok=True)

        tb = os.path.join(src, "tensorboard")
        if os.path.exists(tb):
            shutil.copytree(tb, os.path.join(dst, "tensorboard"))

In [27]:
!nvidia-smi > /content/submission/gpu_info.txt

In [28]:
import shutil

shutil.make_archive(
    "/content/maxtext_gpu_results",
    "zip",
    "/content/submission"
)

'/content/maxtext_gpu_results.zip'

In [29]:
!find /content/maxtext_outputs -name "events.out.tfevents*"

/content/maxtext_outputs/qwen_1p7b_gpu_a100_50steps/tensorboard/qwen_1p7b_gpu_a100_50steps/events.out.tfevents.1781903197.8b4ffd166c8f
/content/maxtext_outputs/qwen_0p6b_gpu_a100_50steps/tensorboard/qwen_0p6b_gpu_a100_50steps/events.out.tfevents.1781904343.8b4ffd166c8f
/content/maxtext_outputs/test_a100/tensorboard/test_a100/events.out.tfevents.1781902922.8b4ffd166c8f
